# AI-Powered HR Assistant

This project builds a question-answering assistant for the Nestle HR policy document. It extracts text from the PDF, divides it into overlapping chunks, creates embeddings with OpenAI, and stores them in a persistent Chroma vector database. For each question, the system retrieves relevant passages and provides them as context to GPT-4o, which is instructed to answer from the document and say when information is unavailable.

A Gradio interface presents the query input and generated answer side by side, making it easy to ask questions about the policy.

In [103]:
import os
import json
import httpx2
import gradio as gr
import time

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

http_client = httpx2.Client(headers={"Accept-Encoding": "gzip, deflate"})

In [ ]:
# Load config from JSON
with open("./config.json", "r") as f:
    config = json.load(f)
# Extract OpenAI API key
openai_api_key = config["openai"]["api_key"]
# Set as environment variable (recommended)
os.environ["OPENAI_API_KEY"] = openai_api_key # key can be hard coded here .
model = ChatOpenAI(model="gpt-4o", http_client=http_client)

In [ ]:
## Load the documents
pdf_files = [
    "./the_nestle_hr_policy_pdf_2012.pdf"
    #"./Insurance/Terms and Conditions for Private Car_2.pdf",
    # "./Books/deep-learning-material-dept-ece-amr-blr-1.pdf"
]

documents = []

for file in pdf_files:
    loader = PyPDFLoader(file)
    docs = loader.load()
    documents.extend(docs)

print("Loaded pages:", len(documents))


Loaded pages: 8


In [106]:
## Split the docs into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)
print("Chunks:", len(chunks))

Chunks: 20


In [107]:
## Load the chunks into vector DB
# 1. Initialize the embedding model (Make sure OPENAI_API_KEY is set in your environment variables)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", http_client=http_client)
# 2. Create and persist the Chroma vector database
vector_db = Chroma(
    collection_name="my_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db",  # Where to save data locally, remove if not necessary
)

print("Vectob DB created")

vector_db.add_documents(chunks)

Vectob DB created


['e6dfc708-d297-4268-bfd9-1929acc8cfbe',
 '2ef88c48-feec-492c-9db3-0fa936cf1356',
 '8cca1670-e4ba-4d86-9299-ee9e0f1f04fa',
 'c9536361-a06a-48e9-97de-0547fb7ab235',
 '10d2aacf-e71d-4d6c-9241-ace427038a43',
 '5deea5b6-a7e6-4f54-a327-247bf5fd74d5',
 'a363b2fb-63f8-40c3-b4a0-acd2695f1230',
 'de73239f-d1c1-4990-968b-46374d755c1c',
 '2beef095-d02d-487d-b9d3-7a58ae727cdb',
 '4dd55ac6-bceb-4546-8991-e729c75972ac',
 '055fb4c4-f33c-4c9c-8515-d457c95ab221',
 'ca3d7a76-2266-4185-8d0f-78a1778498f9',
 '81bcc197-38d5-4db4-a49c-0f35c5093ecf',
 '320beecc-8ea9-4588-b765-ca46af729351',
 'cf54ada9-45e7-4883-ad14-658fcfb0307c',
 '95767de8-57ac-4312-a74a-790141c6a32a',
 '40d9e05f-d50e-4564-8f26-a6399b059c03',
 'c0ee2560-b817-40a2-96f7-31c622907178',
 'd9f57ad3-7809-42cd-a025-ecc98e8bd8df',
 '6ea07ecc-b9e9-46a7-9072-d2d381eccc02']

In [108]:
## Retrieve relevant documents based on a query
retriever = vector_db.as_retriever(search_kwargs={"k":3})

In [109]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [110]:
# prompt chain
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.
Use only the context below to answer.
Context:
{context}

Question:
{question}

If answer is not found, say "Not available in documents".
""")


In [111]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
)

In [112]:
query = "Give employment and working conditions at Nestle ?"

answer = rag_chain.invoke(query)

print(answer.content)

Nestlé is committed to providing its employees all over the world with good working conditions, a safe and healthy work environment, and flexible employment possibilities that support a better balance of private and professional life. Additionally, Nestlé encourages employees to have outside interests, especially community involvement. Those with line management responsibilities are required to take personal ownership of safety and health within their area of responsibility and are encouraged to develop their capability in this area. Furthermore, Nestlé cares about all people working inside or outside their premises under contractual obligations with service providers and insists that they take steps to ensure adequate working conditions are available to them.


In [113]:
## Side-by-side HR policy assistant

def query_rag_pipeline(message: str) -> str:
    answer = rag_chain.invoke(message)
    return answer.content


with gr.Blocks(title="Nestle HR Policy Assistant") as demo:
    gr.Markdown("# Nestle HR Policy Assistant")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Query")
            query_box = gr.Textbox(
                label="Ask about the HR policy",
                placeholder="Enter your question...",
                lines=8,
            )
            ask_button = gr.Button("Ask", variant="primary")

        with gr.Column(scale=1):
            gr.Markdown("### Answer")
            answer_box = gr.Markdown("Your answer will appear here.")

    ask_button.click(
        fn=query_rag_pipeline,
        inputs=query_box,
        outputs=answer_box,
    )
    query_box.submit(
        fn=query_rag_pipeline,
        inputs=query_box,
        outputs=answer_box,
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
